# Deliverable 3 — Hold Until Congress Sells

**The real test of congressional edge.** Congress's advantage isn't just *where* they buy — it's *when they exit*. This backtest holds every buy until Congress sells that same position, capturing their actual timing.

- **Entry**: Transaction date (day Congress bought)
- **Exit**: Transaction date of next sale by same member in same ticker (when Congress exited)
- **Benchmark**: SPY from entry to exit (same holding period)
- **Result**: This is how Quiver's tracker shows Congress beating SPY

Outputs → `output/3-hold-until-congress-sells/`: `equity_curve.mp4`, `hold_period_distribution.mp4`, `best_worst_exits.mp4`, `v1_vs_v3_comparison.mp4`, `trade_results.csv`, `summary_stats.json`

In [ ]:
import sys
import json
import warnings
from pathlib import Path
from datetime import timedelta
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve().parent
load_dotenv(REPO_ROOT / '.env')
sys.path.insert(0, str(REPO_ROOT))

from lib import brand, animation, data_quiver, data_massive, backtest
from audio_engine import Cue, render_track, ticks_every

warnings.filterwarnings('ignore')
brand.apply_theme()
print('Imports OK')

In [ ]:
PRICE_END = '2026-07-02'

OUT_DIR = Path('../output/3-hold-until-congress-sells')
OUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR = Path('cache/audio')
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {OUT_DIR.resolve()}')

In [ ]:
# Fetch full Congress trading history (both buys AND sells)
raw_congress = data_quiver.fetch_congress_trades()
buys = data_quiver.clean_congress_buys(raw_congress)
sells = data_quiver.clean_congress_sells(raw_congress)

print(f'{len(buys):,} buy transactions | {len(sells):,} sell transactions')
print(f'Buys: {buys["Ticker"].nunique()} tickers, {buys["Representative"].nunique()} reps')
print(f'Sells: {sells["Ticker"].nunique()} tickers, {sells["Representative"].nunique()} reps')

In [ ]:
# Price cache (shared with prior deliverables — cache hits only)
fetch_start = (buys['TransactionDate'].min() - timedelta(days=5)).strftime('%Y-%m-%d')
tickers = ['SPY'] + sorted(set(buys['Ticker'].unique()) | set(sells['Ticker'].unique()))
price_cache = data_massive.load_price_cache(tickers, fetch_start, PRICE_END)
spy_prices = price_cache['SPY']
print(f'SPY: {len(spy_prices)} trading days')

In [ ]:
# v3: Buy on transaction date, hold until Congress sells
v3_results = backtest.run_backtest_hold_until_sell(
    buys, sells, price_cache, spy_prices,
    entry='transaction', allow_open=False
)

print(f'\nMatching statistics:')
print(f'  Avg hold period: {v3_results["HoldDays"].mean():.0f} days')
print(f'  Min/Max hold: {v3_results["HoldDays"].min():.0f} / {v3_results["HoldDays"].max():.0f} days')
print(f'  Median hold: {v3_results["HoldDays"].median():.0f} days')

v3_results.head(3)

In [ ]:
v3_curve = backtest.equity_curve(v3_results, price_cache, spy_prices)
pol_stats = backtest.politician_stats(v3_results, min_trades=5)

top5 = v3_results.nlargest(5, 'TradeReturn')
bot5 = v3_results.nsmallest(5, 'TradeReturn')
best = top5.iloc[0]

summary = backtest.summarize(v3_results, v3_curve)
summary.update({
    'engine': 'v3_hold_until_congress_sells',
    'avg_hold_days': float(v3_results['HoldDays'].mean()),
    'median_hold_days': float(v3_results['HoldDays'].median()),
    'portfolio_model': 'equal-weight daily rebalance across open positions',
    'top_politician': str(pol_stats.iloc[0]['Representative']),
    'top_politician_return': float(pol_stats.iloc[0]['AvgTradeReturn']),
    'best_trade': {
        'representative': str(best['Representative']),
        'ticker': str(best['Ticker']),
        'buy_date': str(best['BuyDate'].date()),
        'sell_date': str(best['SellDate'].date()),
        'hold_days': int(best['HoldDays']),
        'return': float(best['TradeReturn']),
    },
})

v3_results.to_csv(OUT_DIR / 'trade_results.csv', index=False)
top5.to_csv(OUT_DIR / 'top5_trades.csv', index=False)
bot5.to_csv(OUT_DIR / 'bottom5_trades.csv', index=False)
pol_stats.to_csv(OUT_DIR / 'politician_ranking.csv', index=False)
v3_curve.to_csv(OUT_DIR / 'equity_curve_data.csv', index=False)
with open(OUT_DIR / 'summary_stats.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"${backtest.INITIAL_CAPITAL:,} → portfolio ${summary['final_portfolio']:,.0f} "
      f"| SPY ${summary['final_spy']:,.0f}")
print(f"Top politician: {summary['top_politician']} ({summary['top_politician_return']:.2%} avg)")
print(f"Best trade: {best['Representative']} — {best['Ticker']} "
      f"({best['HoldDays']:.0f} days held) → {best['TradeReturn']:.1%}")

In [ ]:
# Visual 1: Equity curve — hold until Congress sells
DRAW_S, HOLD_S = 12.0, 2.0
wav = render_track(
    ticks_every('tick', 1.0, DRAW_S - 1.0, 2.0, gain_db=-8)
    + [Cue('line_swell', DRAW_S - 1.8), Cue('data_blip', DRAW_S + 0.4, gain_db=-4)],
    duration_s=DRAW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd3_equity.wav',
)

animation.animate_lines(
    series=[
        {'label': 'Hold Until Congress Sells', 'short': 'Congress', 'dates': v3_curve['Date'].values,
         'values': v3_curve['CumPortfolio'].values, 'color': brand.GREEN},
        {'label': 'S&P 500 Buy & Hold', 'short': 'SPY', 'dates': v3_curve['Date'].values,
         'values': v3_curve['CumSPY'].values, 'color': brand.OFF_WHITE, 'alpha': 0.9},
    ],
    title='COPY CONGRESS EXITS - HOLD UNTIL THEY SELL',
    output_path=OUT_DIR / 'equity_curve.mp4',
    draw_seconds=DRAW_S, hold_seconds=HOLD_S,
    ref_line=backtest.INITIAL_CAPITAL,
    audio_wav=wav,
)

In [ ]:
# Visual 2: Distribution of holding periods
import pandas as pd
import numpy as np

bins = [0, 30, 60, 90, 180, 365, 1000]
labels = ['0-30d', '31-60d', '61-90d', '91-180d', '181d-1y', '>1y']
hold_dist = pd.cut(v3_results['HoldDays'], bins=bins, labels=labels)
counts = hold_dist.value_counts().sort_index()

GROW_S, HOLD_S = 1.5, 2.5
wav = render_track(
    [Cue('bar_grow', 0.1)] + ticks_every('tick', GROW_S, GROW_S + 0.7, 0.08, gain_db=-10)
    + [Cue('data_blip', GROW_S + 1.0, gain_db=-4)],
    duration_s=GROW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd3_holdperiod.wav',
)

animation.animate_bars(
    labels=list(counts.index),
    values=counts.tolist(),
    colors=[brand.GREEN] * len(counts),
    title='HOW LONG DID CONGRESS HOLD EACH POSITION?',
    output_path=OUT_DIR / 'hold_period_distribution.mp4',
    grow_seconds=GROW_S, hold_seconds=HOLD_S,
    annotations=[str(v) for v in counts.tolist()],
    audio_wav=wav,
)

In [ ]:
# Visual 3: Top 5 best and worst exits (Congress's exit timing)
def exit_label(row):
    last = str(row['Representative']).split()[-1]
    return f"{row['Ticker']}\n{last}\n{row['HoldDays']:.0f}d"

combined = pd.concat([top5, bot5])
GROW_S, HOLD_S = 1.5, 2.5
wav = render_track(
    [Cue('bar_grow', 0.1), Cue('data_blip', GROW_S, gain_db=-4),
     Cue('error_buzz', GROW_S + 0.5, gain_db=-10)],
    duration_s=GROW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd3_topbottom.wav',
)

animation.animate_bars(
    labels=[exit_label(r) for _, r in combined.iterrows()],
    values=combined['TradeReturn'].tolist(),
    colors=[brand.GREEN if r >= 0 else brand.RED for r in combined['TradeReturn']],
    title='TOP 5 & BOTTOM 5 EXITS - CONGRESS HOLDS UNTIL SALE',
    output_path=OUT_DIR / 'best_worst_exits.mp4',
    grow_seconds=GROW_S, hold_seconds=HOLD_S,
    divider_after=5, group_labels=('TOP 5', 'BOTTOM 5'),
    audio_wav=wav,
)

In [ ]:
# Visual 4: The difference between fixed 60-day hold (v1) vs hold-until-sell (v3)
# This shows the value of Congress's exit timing
import matplotlib.pyplot as plt
import matplotlib.animation as mpl_animation
from matplotlib.ticker import FuncFormatter

v1_results = backtest.run_backtest(
    buys, price_cache, spy_prices,
    hold_days=60, entry='transaction', verbose=False
)
v1_curve = backtest.equity_curve(v1_results, price_cache, spy_prices)

def animate_v1_v3(v1c, v3c, output_path, draw_s=12.0, hold_s=2.5, audio_wav=None):
    n_draw = int(draw_s * brand.VIDEO_FPS)
    n_total = n_draw + int(hold_s * brand.VIDEO_FPS)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(brand.FIG_W, brand.FIG_H), facecolor=brand.BG)
    fig.suptitle('THE VALUE OF CONGRESS\'S EXIT TIMING', color=brand.OFF_WHITE,
                 fontsize=22, fontfamily=brand.VCR, fontweight='bold', y=0.97)

    y_lo = min(v1c['CumPortfolio'].min(), v1c['CumSPY'].min(),
               v3c['CumPortfolio'].min(), v3c['CumSPY'].min()) * 0.95
    y_hi = max(v1c['CumPortfolio'].max(), v1c['CumSPY'].max(),
               v3c['CumPortfolio'].max(), v3c['CumSPY'].max()) * 1.08

    panels = []
    for ax, curve, color, subtitle in [
        (ax1, v1c, brand.OLIVE, 'v1 - FIXED 60-DAY HOLD'),
        (ax2, v3c, brand.GREEN, 'v3 - HOLD UNTIL CONGRESS SELLS'),
    ]:
        ax.set_facecolor(brand.BG)
        ax.set_title(subtitle, color=color, fontsize=13, pad=12, fontfamily=brand.VCR)
        ax.grid(True, color=brand.GRID, linewidth=0.6, alpha=0.6, zorder=0)
        ax.set_axisbelow(True)
        for s in ax.spines.values():
            s.set_edgecolor(brand.GRID)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        dates = curve['Date'].values
        ax.set_xlim(dates[0], dates[-1])
        ax.set_ylim(y_lo, y_hi)
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax.tick_params(labelsize=9)
        ax.axhline(backtest.INITIAL_CAPITAL, color=brand.GRID, linewidth=0.8,
                   linestyle='--', alpha=0.7)
        ln_spy, = ax.plot([], [], color=brand.OFF_WHITE, linewidth=2, alpha=0.85, label='SPY')
        ln_strat, = ax.plot([], [], color=color, linewidth=2.5, label='Strategy')
        ax.legend(facecolor=brand.BG, edgecolor=brand.GRID, labelcolor=brand.OFF_WHITE,
                  loc='upper left', prop={'family': brand.VCR, 'size': 11})
        val_txt = ax.text(0.97, 0.95, '', transform=ax.transAxes, color=color,
                          fontsize=14, ha='right', va='top', fontfamily=brand.VCR)
        idx = np.linspace(0, len(curve) - 1, n_draw, dtype=int)
        panels.append((curve, ln_strat, ln_spy, val_txt, idx))

    def update(frame):
        f = min(frame, n_draw - 1)
        artists = []
        for curve, ln_strat, ln_spy, val_txt, idx in panels:
            i = idx[f]
            d = curve['Date'].values[:i + 1]
            ln_strat.set_data(d, curve['CumPortfolio'].values[:i + 1])
            ln_spy.set_data(d, curve['CumSPY'].values[:i + 1])
            val_txt.set_text(f"${curve['CumPortfolio'].iloc[i]:,.0f}")
            artists += [ln_strat, ln_spy, val_txt]
        return artists

    ani = mpl_animation.FuncAnimation(fig, update, frames=n_total,
                                      interval=1000 / brand.VIDEO_FPS, blit=True)
    print(f'Rendering {output_path.name}  ({n_total} frames)...')
    animation.save_animation(ani, fig, output_path, audio_wav)


DRAW_S, HOLD_S = 12.0, 2.5
wav = render_track(
    ticks_every('tick', 1.0, DRAW_S - 1.0, 2.0, gain_db=-8)
    + [Cue('line_swell', DRAW_S - 1.8), Cue('data_blip', DRAW_S + 0.4, gain_db=-4)],
    duration_s=DRAW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd3_v1v3.wav',
)

animate_v1_v3(v1_curve, v3_curve, OUT_DIR / 'v1_vs_v3_comparison.mp4',
              draw_s=DRAW_S, hold_s=HOLD_S, audio_wav=wav)

In [ ]:
# Summary comparison
print('\n' + '='*70)
print('SUMMARY: v1 (60-day fixed) vs v3 (hold until Congress sells)')
print('='*70)

v1_summary = backtest.summarize(v1_results, v1_curve)

print(f'\nExcess Return (vs SPY):')
print(f'  v1 (60-day): {v1_summary["avg_excess_return"]:+.2%}')
print(f'  v3 (hold): {summary["avg_excess_return"]:+.2%}')
print(f'  Improvement: {summary["avg_excess_return"] - v1_summary["avg_excess_return"]:+.2%}')

print(f'\nPortfolio Growth:')
print(f'  v1 final: ${v1_summary["final_portfolio"]:,.0f}')
print(f'  v3 final: ${summary["final_portfolio"]:,.0f}')
print(f'  SPY final: ${summary["final_spy"]:,.0f}')

print(f'\nBeat SPY Rate:')
print(f'  v1: {v1_summary["beat_spy_rate"]:.1%} of trades')
print(f'  v3: {summary["beat_spy_rate"]:.1%} of trades')